In [1]:
!pip install torch torchvision matplotlib -q

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)

PyTorch Version: 2.11.0+cpu


In [2]:
a = torch.tensor([1, 2, 3, 4])
b = torch.tensor([[1, 2], [3, 4]])

print("1D Tensor:", a)
print("2D Tensor:", b)
print("Shape:", b.shape)

random_tensor = torch.rand(3, 3)
print("Random Tensor:\n", random_tensor)

1D Tensor: tensor([1, 2, 3, 4])
2D Tensor: tensor([[1, 2],
        [3, 4]])
Shape: torch.Size([2, 2])
Random Tensor:
 tensor([[0.8711, 0.0250, 0.8807],
        [0.9319, 0.2334, 0.7509],
        [0.8492, 0.1401, 0.4030]])


In [3]:
x = torch.tensor(5.0, requires_grad=True)

y = x**2 + 3*x + 2

y.backward()

print("Output:", y.item())
print("Gradient:", x.grad.item())

Output: 42.0
Gradient: 13.0


In [4]:
# Generate sample data
X = torch.randn(100, 1)
y = 3 * X + 2

# Define model
model = nn.Linear(1, 1)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
for epoch in range(100):
    predictions = model(X)
    loss = criterion(predictions, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

print("Learned Weight:", model.weight.item())
print("Learned Bias:", model.bias.item())

Epoch 20, Loss: 5.9906
Epoch 40, Loss: 2.5270
Epoch 60, Loss: 1.0683
Epoch 80, Loss: 0.4527
Epoch 100, Loss: 0.1924
Learned Weight: 2.612853765487671
Learned Bias: 1.8403687477111816


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

print("Training Samples:", len(train_dataset))
print("Testing Samples:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 161kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.9MB/s]

Training Samples: 60000
Testing Samples: 10000


In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

model = CNN()
print(model)

CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    total_loss = 0

    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.2524
Epoch 2, Loss: 0.0644
Epoch 3, Loss: 0.0457


In [8]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.56%


In [9]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

data = np.sin(np.linspace(0, 100, 1000))

X = []
y = []

sequence_length = 20

for i in range(len(data) - sequence_length):
    X.append(data[i:i+sequence_length])
    y.append(data[i+sequence_length])

X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(-1)

print("Input Shape:", X.shape)
print("Output Shape:", y.shape)

Input Shape: torch.Size([980, 20, 1])
Output Shape: torch.Size([980, 1])


In [10]:
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            batch_first=True
        )

        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        output, hidden = self.rnn(x)
        return self.fc(output[:, -1, :])

model = RNNModel()
print(model)

RNNModel(
  (rnn): RNN(1, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [11]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(20):
    output = model(X)
    loss = criterion(output, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")

Epoch 5, Loss: 0.397665
Epoch 10, Loss: 0.324868
Epoch 15, Loss: 0.249469
Epoch 20, Loss: 0.163857


In [12]:
class SequenceModel(nn.Module):
    def __init__(self, model_type="LSTM"):
        super().__init__()

        if model_type == "RNN":
            self.layer = nn.RNN(1, 32, batch_first=True)

        elif model_type == "LSTM":
            self.layer = nn.LSTM(1, 32, batch_first=True)

        elif model_type == "GRU":
            self.layer = nn.GRU(1, 32, batch_first=True)

        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        output, hidden = self.layer(x)
        return self.fc(output[:, -1, :])

for name in ["RNN", "LSTM", "GRU"]:
    model = SequenceModel(name)
    print(name, "Model Created Successfully")

RNN Model Created Successfully
LSTM Model Created Successfully
GRU Model Created Successfully


In [13]:
class RegularizedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)

        self.dropout = nn.Dropout(0.3)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)

        self.fc3 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)

        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)

        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        return self.fc3(x)

model = RegularizedModel()
print(model)

RegularizedModel(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)


In [14]:
from torch.utils.data import random_split, DataLoader
import torchvision
import torchvision.transforms as transforms

dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

train_data, val_data = random_split(
    dataset, [50000, 10000]
)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128)

In [15]:
import copy

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_loss = float('inf')
patience = 3
counter = 0

for epoch in range(20):

    model.train()
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}, Validation Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early Stopping Activated!")
        break

model.load_state_dict(best_weights)

Epoch 1, Validation Loss: 0.1446
Epoch 2, Validation Loss: 0.1111
Epoch 3, Validation Loss: 0.0865
Epoch 4, Validation Loss: 0.0851
Epoch 5, Validation Loss: 0.0837
Epoch 6, Validation Loss: 0.0775
Epoch 7, Validation Loss: 0.0808
Epoch 8, Validation Loss: 0.0762
Epoch 9, Validation Loss: 0.0774
Epoch 10, Validation Loss: 0.0758
Epoch 11, Validation Loss: 0.0781
Epoch 12, Validation Loss: 0.0785
Epoch 13, Validation Loss: 0.0776
Early Stopping Activated!


<All keys matched successfully>

In [16]:
import torch
import torch.nn as nn
import torchvision.models as models

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
model.fc = nn.Linear(model.fc.in_features, 10)

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 62.6MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [20]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models

In [21]:
digits = load_digits()

X = digits.images
y = digits.target

print("Dataset loaded successfully!")
print("Images:", X.shape)
print("Labels:", y.shape)

Dataset loaded successfully!
Images: (1797, 8, 8)
Labels: (1797,)


In [22]:
digits = load_digits()

X = digits.images
y = digits.target

X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
X = X.repeat(1, 3, 1, 1)

X = torch.nn.functional.interpolate(
    X, size=(64, 64), mode='bilinear'
)

X = X / 16.0

y = torch.tensor(y, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=32
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [23]:
model = models.mobilenet_v2(weights="DEFAULT")

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, 10
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("MobileNetV2 Loaded Successfully!")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 36.3MB/s]


MobileNetV2 Loaded Successfully!


In [24]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

for epoch in range(3):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1,
          "Loss:", total_loss/len(train_loader))

Epoch: 1 Loss: 1.879814076423645
Epoch: 2 Loss: 1.2817827357186211
Epoch: 3 Loss: 1.0335397799809773


In [25]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predicted = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total, "%")

Test Accuracy: 72.5 %


In [26]:
!pip install transformers torch -q

In [27]:
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

print("Model Loaded Successfully!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Model Loaded Successfully!


In [28]:
texts = [
    "I love learning artificial intelligence.",
    "This project is very interesting.",
    "I am disappointed with the results.",
    "Deep learning is difficult but exciting."
]

results = sentiment(texts)

for text, result in zip(texts, results):
    print("Text:", text)
    print("Sentiment:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("--------------------------")

Text: I love learning artificial intelligence.
Sentiment: POSITIVE
Confidence: 0.9995
--------------------------
Text: This project is very interesting.
Sentiment: POSITIVE
Confidence: 0.9998
--------------------------
Text: I am disappointed with the results.
Sentiment: NEGATIVE
Confidence: 0.9998
--------------------------
Text: Deep learning is difficult but exciting.
Sentiment: POSITIVE
Confidence: 0.9981
--------------------------


In [29]:
!pip install transformers torch -q

In [30]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("Language Model Loaded!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Language Model Loaded!


In [31]:
prompt = "Artificial Intelligence is transforming the world because"

result = generator(
    prompt,
    max_new_tokens=60,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    pad_token_id=generator.tokenizer.eos_token_id
)

print(result[0]["generated_text"])

Artificial Intelligence is transforming the world because of its ability to understand the world from its own perspective.





The Artificial Intelligence (AI) is a global phenomenon that is evolving from a simple observation of the world of the world to a more complex and complex and complex.
AI is a multi-disciplinary field of research


In [33]:
!pip install transformers==4.44.2 -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
